# Citify Phase F v4 — Reinfolib 全自治体拡張 (地方分割版)

全国 1794 市区町村への Reinfolib 5 API バッチを **9 地方** に分割実行する。
ローカル PC と Colab を並行で動かして、最短で全国カバレッジを完成させる。

## 地方区分 (9 region)
| # | region | 都道府県 | 自治体目安 | 推定時間 |
|---|---|---|---|---|
| 1 | `hokkaido_tohoku` | 北海道 + 東北 6 県 | ~230 | 6-8h |
| 2 | `kanto` | 茨城-神奈川 (1 都 6 県) | ~310 | 9-11h |
| 3 | `koshinetsu` | 新潟・山梨・長野 | ~100 | 3-4h |
| 4 | `hokuriku` | 富山・石川・福井 | ~50 | 2h |
| 5 | `tokai` | 岐阜-三重 | ~170 | 5-6h |
| 6 | `kinki` | 滋賀-和歌山 | ~200 | 6-7h |
| 7 | `chugoku` | 鳥取-山口 | ~110 | 3-4h |
| 8 | `shikoku` | 徳島-高知 | ~95 | 3h |
| 9 | `kyushu_okinawa` | 福岡-沖縄 | ~270 | 8-10h |

## 並行実行戦略
1 セッション 1 region 担当 (Colab は最大 12h、長い region は Pro+ 24h 推奨)。**地方を分担**:

| 担当 | 地方順 |
|---|---|
| **ローカル PC** | hokkaido_tohoku → kanto → koshinetsu → hokuriku |
| **Colab** | tokai → kinki → chugoku → shikoku → kyushu_okinawa |

Colab セッション切れ時は `RESUME=True` で続きから再開可。

## 使い方
1. Colab の `🔑 Secrets` で `REINFOLIB_API_KEY` を登録
2. Cell 5 で `REGION` を変える (`tokai` 等)
3. Cell 1-7 を順に実行 (放置で OK)
4. 完了 CSV (`reinfolib_normalized_{REGION}.csv`) は Drive に保存
5. ローカルで DL → `bq` ロード (Cell 8 参照)

## Cell 1: パッケージインストール

In [ ]:
!pip install -q httpx pandas tqdm geopandas pyogrio

## Cell 2: Drive mount + API キー設定

In [ ]:
import os
from pathlib import Path

from google.colab import drive  # type: ignore
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/citify')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'work dir: {WORK_DIR}')

from google.colab import userdata  # type: ignore
os.environ['REINFOLIB_API_KEY'] = userdata.get('REINFOLIB_API_KEY').strip()
print('REINFOLIB_API_KEY:', 'OK' if os.environ.get('REINFOLIB_API_KEY') else 'NOT SET')

## Cell 3: Citify リポジトリ clone (regions.py + build_targets_full.py を含む最新版)

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('/content/citify')
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/yujmatsu/citify.git /content/citify
else:
    !cd /content/citify && git pull --rebase --autostash

sys.path.insert(0, str(REPO_DIR))

from scrapers.reinfolib.client import ReinfolibClient, ReinfolibAPIError
from scrapers.reinfolib.parsers.xit001 import aggregate_used_apartments
from scrapers.reinfolib.parsers.xgt001 import aggregate_shelters
from scrapers.reinfolib.parsers.xkt013 import aggregate_future_population
from scrapers.reinfolib.parsers.xkt010 import aggregate_medical
from scrapers.reinfolib.parsers.xkt007 import aggregate_childcare
from scrapers.reinfolib.regions import REGION_MAP, REGION_LABELS, is_in_region, list_regions

print('imports OK, regions:', list_regions())

## Cell 4: targets_full.csv 生成 (N03 → centroid、初回のみ ~10 分)

Drive に既に保存済の場合はスキップ。

In [ ]:
TARGETS_FULL_CSV = WORK_DIR / 'reinfolib_targets_full.csv'

if TARGETS_FULL_CSV.exists():
    print(f'既存ファイル使用: {TARGETS_FULL_CSV}')
else:
    print('N03 から targets_full.csv を生成中... (~10 分)')
    import subprocess
    result = subprocess.run(
        [
            'python', '-m', 'scripts.build_reinfolib_targets_full',
            '--master', str(REPO_DIR / 'infra/seed/municipality_master.csv'),
            '--output', str(TARGETS_FULL_CSV),
            '--work-dir', '/content/n03_work',
            '-v',
        ],
        cwd=str(REPO_DIR / 'apps/api'),
        capture_output=True, text=True,
    )
    print('stdout:', result.stdout[-500:])
    print('stderr:', result.stderr[-1000:])
    if result.returncode != 0:
        raise RuntimeError('build_reinfolib_targets_full failed')

import pandas as pd
df_targets_all = pd.read_csv(TARGETS_FULL_CSV, dtype={'municipality_code': str})
df_targets_all['municipality_code'] = df_targets_all['municipality_code'].str.zfill(5)
print(f'全 targets: {len(df_targets_all)} 自治体')
df_targets_all.head()

## Cell 5: REGION 選択 (このセッションで担当する地方を 1 つ)

実行前に下の `REGION` を変更してください。利用可能な値:
- `'hokkaido_tohoku'` `'kanto'` `'koshinetsu'` `'hokuriku'`
- `'tokai'` `'kinki'` `'chugoku'` `'shikoku'` `'kyushu_okinawa'`

In [ ]:
REGION = 'tokai'      # ← ここを変更
RESUME = True          # 既存 CSV があれば続きから (recommended)
RATE_LIMIT_SEC = 1.0   # Reinfolib 推奨

assert REGION in REGION_MAP, f'unknown REGION={REGION}, valid={list(REGION_MAP)}'
print(f'担当 REGION: {REGION} ({REGION_LABELS[REGION]})')
print(f'  都道府県 prefix: {REGION_MAP[REGION]}')

df_targets = df_targets_all[
    df_targets_all['municipality_code'].apply(lambda c: is_in_region(c, REGION))
].reset_index(drop=True)
print(f'  対象自治体: {len(df_targets)} 件')
df_targets.head()

## Cell 6: バッチ実行 (resume 対応、進捗バー付き)

出力 CSV は `Drive/citify/reinfolib_normalized_{REGION}.csv` に append。途中で Colab セッションが切れても、再起動 → Cell 1-5 → Cell 6 で続きから再開できる。

In [ ]:
import datetime as dt
import csv
import logging
from tqdm.auto import tqdm

logging.basicConfig(level=logging.WARNING)

OUTPUT_CSV = WORK_DIR / f'reinfolib_normalized_{REGION}.csv'
OUTPUT_COLUMNS = [
    'municipality_code',
    'used_apartment_median_price_man_yen', 'used_apartment_sample_size',
    'used_apartment_median_unit_price_yen', 'used_apartment_avg_building_age',
    'emergency_shelter_count', 'emergency_shelter_official_link',
    'population_2025_estimated', 'population_2050_estimated', 'population_change_2025_2050_pct',
    'medical_facility_count', 'medical_hospital_count', 'medical_clinic_count',
    'childcare_facility_count', 'kindergarten_count', 'nursery_count',
    'reinfolib_loaded_at', 'reinfolib_source_url',
]
SOURCE_URL = 'https://www.reinfolib.mlit.go.jp/'

# resume
completed_codes: set[str] = set()
if RESUME and OUTPUT_CSV.exists():
    with OUTPUT_CSV.open('r', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            completed_codes.add(r['municipality_code'])
    print(f'resume: {len(completed_codes)} 自治体は完了済 (skip)')

todo = df_targets[~df_targets['municipality_code'].isin(completed_codes)]
print(f'残: {len(todo)} 自治体 (合計 {len(df_targets)})')


def fetch_one_municipality(client: ReinfolibClient, target: dict) -> dict:
    code = target['municipality_code']
    method = target['xit001_method']
    param = target['xit001_param']
    lat = float(target['center_lat'])
    lng = float(target['center_lng'])
    out = {'municipality_code': code}

    try:
        trades = client.fetch_trades_4quarters(method, param)
        out.update(aggregate_used_apartments(trades))
    except Exception as e:
        print(f'  xit001 fail code={code}: {e}')
        out.update({
            'used_apartment_median_price_man_yen': None, 'used_apartment_sample_size': 0,
            'used_apartment_median_unit_price_yen': None, 'used_apartment_avg_building_age': None,
        })
    try:
        f1 = client.fetch_geojson_tiles('XGT001', lng, lat, z=11, radius=1)
        out.update(aggregate_shelters(f1, lat, lng))
    except Exception as e:
        print(f'  xgt001 fail code={code}: {e}')
        out.update({
            'emergency_shelter_count': 0,
            'emergency_shelter_official_link': f'https://disaportal.gsi.go.jp/hazardmap/maps/index.html?ll={lat},{lng}&z=12',
        })
    try:
        f2 = client.fetch_geojson_tiles('XKT013', lng, lat, z=11, radius=1)
        out.update(aggregate_future_population(f2))
    except Exception as e:
        print(f'  xkt013 fail code={code}: {e}')
        out.update({
            'population_2025_estimated': None, 'population_2050_estimated': None,
            'population_change_2025_2050_pct': None,
        })
    try:
        f3 = client.fetch_geojson_tiles('XKT010', lng, lat, z=13, radius=2)
        out.update(aggregate_medical(f3))
    except Exception as e:
        print(f'  xkt010 fail code={code}: {e}')
        out.update({
            'medical_facility_count': None, 'medical_hospital_count': None, 'medical_clinic_count': None,
        })
    try:
        f4 = client.fetch_geojson_tiles('XKT007', lng, lat, z=13, radius=2)
        if method == 'city_sum':
            out.update(aggregate_childcare(f4, municipality_code=None))
        else:
            out.update(aggregate_childcare(f4, municipality_code=code))
    except Exception as e:
        print(f'  xkt007 fail code={code}: {e}')
        out.update({
            'childcare_facility_count': None, 'kindergarten_count': None, 'nursery_count': None,
        })

    out['reinfolib_loaded_at'] = dt.datetime.now(dt.UTC).isoformat()
    out['reinfolib_source_url'] = SOURCE_URL
    return out


write_header = not OUTPUT_CSV.exists()
with ReinfolibClient(rate_limit_sec=RATE_LIMIT_SEC) as client, OUTPUT_CSV.open('a', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
    if write_header:
        writer.writeheader()
    for _, target in tqdm(todo.iterrows(), total=len(todo), desc=f'fetch {REGION}'):
        try:
            row = fetch_one_municipality(client, target.to_dict())
            writer.writerow(row)
            f.flush()
        except Exception as e:
            print(f'FAIL code={target["municipality_code"]} err={e}')

print(f'done. output: {OUTPUT_CSV}')
print(f'rows: {sum(1 for _ in OUTPUT_CSV.open("r")) - 1}')

## Cell 7: 完了状況 サマリ (任意)

Drive 内の各 region CSV の進捗を一覧表示。

In [ ]:
import csv
from pathlib import Path

print(f'{"region":<20} {"label":<15} {"rows":>6}  status')
print('-' * 65)
for r in list_regions():
    csv_path = WORK_DIR / f'reinfolib_normalized_{r}.csv'
    label = REGION_LABELS[r]
    if csv_path.exists():
        with csv_path.open('r', encoding='utf-8') as f:
            n = sum(1 for _ in f) - 1
        target_count = sum(
            1 for _, t in df_targets_all.iterrows() if is_in_region(t['municipality_code'], r)
        )
        mark = '✅' if n >= target_count else '🔄'
        print(f'{r:<20} {label:<15} {n:>6}  {mark} ({n}/{target_count})')
    else:
        print(f'{r:<20} {label:<15} {0:>6}  ⏳ 未着手')

## Cell 8: ローカルで BQ MERGE 実行する手順 (任意)

完了したら Drive から CSV を DL → ローカル Citify リポジトリで:

```bash
# 1. Drive から CSV を全部 DL (Drive App or browser で MyDrive/citify/ を開く)
#    reinfolib_normalized_{region}.csv を infra/seed/ にコピー

# 2. 一括ロード (region 別に MERGE UPDATE、既存列は保護)
cd /home/yujmatsu/projects/citify/apps/api
for REGION in hokkaido_tohoku kanto koshinetsu hokuriku tokai kinki chugoku shikoku kyushu_okinawa; do
  CSV="../../infra/seed/reinfolib_normalized_${REGION}.csv"
  if [ -f "$CSV" ]; then
    echo "=== loading $REGION ==="
    .venv/bin/python -m scripts.load_reinfolib_stats --input "$CSV"
  fi
done

# 3. 確認 (人口減少率 top 20)
bq query --use_legacy_sql=false "
SELECT municipality_code, municipality_name, prefecture,
       population_change_2025_2050_pct, used_apartment_median_price_man_yen,
       medical_facility_count, childcare_facility_count
FROM \`citify-dev.citify_curated.municipality_stats\`
WHERE population_change_2025_2050_pct IS NOT NULL
ORDER BY population_change_2025_2050_pct ASC
LIMIT 20
"

# 4. commit + push (CSV は容量大の場合 .gitignore 候補)
cd /home/yujmatsu/projects/citify
git add infra/seed/reinfolib_normalized_*.csv
git commit -m 'feat(plan-a-f-v4): Reinfolib 全自治体拡張 (地方分割バッチ)'
git push origin main
```

## トラブルシューティング

| 症状 | 対処 |
|---|---|
| Colab セッション切れ | RESUME=True で Cell 1-6 再実行 (skip 自動) |
| `quota_exceeded` | Reinfolib 1日上限。24時間後に再開 |
| `xkt010 fail status=400` | 沿岸自治体で centroid が海上に出た可能性。手動 lat/lng 修正 |
| centroid が間違ってる | targets_full.csv の lat/lng を手動修正 → 該当 region の CSV を削除して再実行 |

## 設計
- 1 region 1 CSV (`reinfolib_normalized_{region}.csv`) で並行衝突なし
- BQ MERGE UPDATE-only なので、ローカルと Colab が同じ自治体を 2 回叩いても最新書込が勝つ (害なし)
- ローカル `python -m scrapers.reinfolib fetch-all --region X --resume` と同じ動作 (Notebook はその Colab 版)